In [ ]:
import os
import pandas as pd
import re


try:
    from openpyxl import load_workbook
except ImportError:
    print("Public message removed for release.")
    exit()

In [ ]:

def get_filename_without_extension(xlsx_path):
    
    
    base_name = os.path.basename(xlsx_path)
    
    file_name_without_extension = os.path.splitext(base_name)[0]
    return file_name_without_extension

In [ ]:

def extract_energy_from_out(energy_out_file_path):
    pattern = r'^FINAL SINGLE POINT ENERGY\s+(-?\d+\.\d+)' 
    with open(energy_out_file_path, 'r') as file:
        for line in file:
            match = re.match(pattern, line)
            if match:
                return float(match.group(1))
    return None

In [ ]:
import re

def extract_homo_lumo(energy_out_file_path):
    
    
    with open(energy_out_file_path, 'r') as file:
        lines = file.readlines()
    
    
    start_index = None
    for idx in range(len(lines) - 1, -1, -1):
        if "ORBITAL ENERGIES" in lines[idx]:
            start_index = idx
            break
    
    if start_index is None:
        start_index = 0

    
    
    
    pattern = re.compile(r'^\s*(\d+)\s+([\d\.]+)\s+(-?[\d\.]+)\s+(-?[\d\.]+)')
    
    header_found = False  
    prev_occ = None       
    prev_eV = None        

    
    for line in lines[start_index:]:
        
        if not header_found:
            if "NO" in line and "OCC" in line and "E(Eh)" in line and "E(eV)" in line:
                header_found = True  
            continue  
        
        
        match = pattern.match(line)
        if match:
            
            
            line_no = int(match.group(1))
            occ = float(match.group(2))
            energy_eh = float(match.group(3))  
            energy_ev = float(match.group(4))
            
            
            if prev_occ is None:
                prev_occ = occ
                prev_eV = energy_ev
            else:
                
                if prev_occ != 0.0 and occ == 0.0:
                    homo = prev_eV  
                    lumo = energy_ev 
                    return homo, lumo
                
                prev_occ = occ
                prev_eV = energy_ev

    
    raise ValueError("Public message removed for release.")

In [ ]:
def extract_dipole_moment(opt_out_file_path):
    
    
    
    
    
    
    
    
    
    pattern = re.compile(r"^Magnitude \(Debye\)\s*:\s*([\d\.\-eE]+)")
    
    
    with open(opt_out_file_path, 'r') as file:
        
        for line in file:
            
            match = pattern.match(line)
            if match:
                
                dipole_str = match.group(1)
                try:
                    
                    dipole_value = float(dipole_str)
                    return dipole_value
                except ValueError:
                    
                    return None
    
    return None

In [ ]:

def extract_gibbs_correction(opt_out_file_path):
    
    
    
    
    
    
    
    
    pattern = re.compile(r"G-E\(el\)\s*\.*\s*([\d\.\-]+)\s*Eh", re.IGNORECASE)
    
    
    with open(opt_out_file_path, 'r') as file:
        for line in file:
            match = pattern.search(line)
            if match:
                try:
                    gibbs_correction = float(match.group(1))
                    return gibbs_correction
                except ValueError:
                    return None
    
    return None

In [ ]:

def extract_inner_energy_correction(opt_out_file_path):
    
    
    
    
    
    
    
    
    
    
    
    
    
    pattern = re.compile(
        r"^(Zero point energy|Thermal vibrational correction|Thermal rotational correction|Thermal translational correction)\s*\.*\s*([\d\.\-]+)\s*Eh",
        re.IGNORECASE | re.MULTILINE
    )
    
    total_correction = 0.0  
    found = False         
    
    
    with open(opt_out_file_path, 'r') as file:
        content = file.read()
        
        matches = pattern.findall(content)
        
        
        for label, value_str in matches:
            try:
                total_correction += float(value_str)
                found = True
            except ValueError:
                
                continue

    
    return total_correction if found else None

In [ ]:


def extract_enthalpy_correction(opt_out_file_path):
    
    
    
    
    
    
    pattern = re.compile(r"Thermal Enthalpy correction\s*\.*\s*([\d\.\-]+)\s*Eh", re.IGNORECASE)
    
    
    with open(opt_out_file_path, 'r') as file:
        for line in file:
            match = pattern.search(line)
            if match:
                try:
                    enthalpy_correction = float(match.group(1))
                    return enthalpy_correction
                except ValueError:
                    return None
    
    return None

In [ ]:


def extract_entropy(opt_out_file_path):
    
    
    
    
    
    
    
    
    pattern = re.compile(r"Final entropy term\s*\.*\s*([\d\.\-]+)\s*Eh", re.IGNORECASE)
    
    with open(opt_out_file_path, 'r') as file:
        for line in file:
            match = pattern.search(line)
            if match:
                try:
                    entropy = float(match.group(1))
                    return entropy
                except ValueError:
                    return None
    return None

In [ ]:


def extract_properties_from_energycal(df, energy_success_dir):

    df['Energy (Hatree)'] = 0.0 
    df['HOMO (eV)'] = 0.0 
    df['LUMO (eV)'] = 0.0 
    
    
    for filename in os.listdir(energy_success_dir):
        if filename.endswith('.out'):
            
            mol_name = filename.rsplit('.', 1)[0]
            
            energy = extract_energy_from_out(os.path.join(energy_success_dir, filename)) 
            homo, lumo = extract_homo_lumo(os.path.join(energy_success_dir, filename)) 

            
            energy = float(energy) if energy else 0.0
            homo = float(homo) if homo else 0.0
            lumo = float(lumo) if lumo else 0.0
            
            
            
            df.loc[df['Name'] == mol_name, 'Energy (Hatree)'] = float(energy)
            df.loc[df['Name'] == mol_name, 'HOMO (eV)'] = homo
            df.loc[df['Name'] == mol_name, 'LUMO (eV)'] = lumo

    return df

In [ ]:

def extract_properties_from_opt_freq(df, opt_success_dir):
    
    df["Inner energy correction (Hatree)"] = 0.0 
    df["Thermal correction to Enthalpy (Hatree)"] = 0.0 
    df["Thermal correction to Gibbs Free Energy (Hatree)"] = 0.0 
    df['Entropy (Hatree)'] = 0.0 
    df['Dipole (Debye)'] = 0.0 
    
    
    for filename in os.listdir(opt_success_dir):
        if filename.endswith('.out'):
            
            mol_name = filename.rsplit('.', 1)[0]
            
            inner_energy_correction = extract_inner_energy_correction(os.path.join(opt_success_dir, filename)) 
            gibbs_correction = extract_gibbs_correction(os.path.join(opt_success_dir, filename)) 
            enthalpy_correction = extract_enthalpy_correction(os.path.join(opt_success_dir, filename)) 
            entropy = extract_entropy(os.path.join(opt_success_dir, filename)) 
            dipole = extract_dipole_moment(os.path.join(opt_success_dir, filename)) 
            
            
            dipole = float(dipole) if dipole else 0.0
            
            df.loc[df['Name'] == mol_name, 'Inner energy correction (Hatree)'] = inner_energy_correction
            df.loc[df['Name'] == mol_name, 'Thermal correction to Gibbs Free Energy (Hatree)'] = gibbs_correction
            df.loc[df['Name'] == mol_name, 'Thermal correction to Enthalpy (Hatree)'] = enthalpy_correction
            df.loc[df['Name'] == mol_name, 'Entropy (Hatree)'] = entropy
            df.loc[df['Name'] == mol_name, 'Dipole (Debye)'] = dipole
                   
    return df

In [ ]:

def data_processing(df):

    df["Gibbs Free Energy (Hatree)"] = df['Energy (Hatree)'] + df["Thermal correction to Gibbs Free Energy (Hatree)"] 
    df["Enthalpy (Hatree)"] = df['Energy (Hatree)'] + df["Inner energy correction (Hatree)"] + df["Thermal correction to Enthalpy (Hatree)"] 
    df["HOMO LUMO Gap (eV)"] = (df['LUMO (eV)'] - df["HOMO (eV)"]) 
    
    
    return df

In [ ]:

opt_success_dir = "opt+freq/success"
energy_success_dir = "energy/success"


df = pd.read_excel("Result.xlsx")


df = extract_properties_from_energycal(df, energy_success_dir)
df = extract_properties_from_opt_freq(df, opt_success_dir)
df = data_processing(df)

In [ ]:
df.to_excel("Result.xlsx", index = None)
df

In [ ]:
import os               
import shutil           
import zipfile          

def collect_and_compress_files(source_dir='.', cleanup=True):
    
    
    source_dir = os.path.abspath(source_dir)                              
    all_results_dir = os.path.join(source_dir, 'all_results')             
    zip_path = os.path.join(source_dir, 'all_results.zip')                
    targets = ['opt+freq', 'energy']                              

    
    if os.path.isdir(all_results_dir):                                    
        shutil.rmtree(all_results_dir)
    os.makedirs(all_results_dir, exist_ok=True)                           
    if os.path.exists(zip_path):                                          
        os.remove(zip_path)

    
    for name in targets:                                                  
        src = os.path.join(source_dir, name)                              
        dst = os.path.join(all_results_dir, name)                         
        if os.path.isdir(src):                                            
            shutil.copytree(src, dst)                                     
        else:
            print("Public message removed for release.")                    

    
    with zipfile.ZipFile(zip_path, mode='w', compression=zipfile.ZIP_DEFLATED) as zf:  
        for root, _, files in os.walk(all_results_dir):                   
            for f in files:                                               
                fpath = os.path.join(root, f)                             
                arcname = os.path.relpath(fpath, all_results_dir)         
                zf.write(fpath, arcname)                                  

    
    if cleanup:                                                           
        shutil.rmtree(all_results_dir)

    print("Public message removed for release.")
    if cleanup:
        print("Public message removed for release.")


collect_and_compress_files(source_dir='.', cleanup=True)  